# 🚦 Streetlight Detection — Full Pipeline & Analysis
**Classes:** `onstreetlight` | `offstreetlight`  
**Covers:** Preprocessing · YOLO · MobileNet SSDLite · Pruning · Iterative Pruning · Quantization · Knowledge Distillation · RPi Export · Full Comparative Analysis

---
> 📷 **Dataset note:** Nighttime, low-light, motion-blurred, handheld images. Preprocessing pipeline is tuned for this.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📦 Cell 1 — Install Dependencies

In [ ]:
# Run once, comment out after
!pip install ultralytics
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install onnx onnxruntime opencv-python albumentations pandas matplotlib seaborn scipy scikit-learn -q
print("All dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 63.9 MB/s eta 0:00:00
All dependencies installed


## 📚 Cell 2 — Imports

In [ ]:
import os, copy, time, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.utils.prune as prune
import torchvision
from torchvision import transforms
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models.detection.ssdlite import SSDLiteHead
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torch.utils.data import DataLoader, Dataset
from torchvision.ops import box_iou
from ultralytics import YOLO
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,
                     'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
sns.set_palette("husl")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch  : 2.10.0+cu128
CUDA     : True


## ⚙️ Cell 3 — Global Configuration

In [ ]:
CFG = dict(
    data_yaml    = "data.yaml",
    img_dir      = "drive/MyDrive/datasetfinal/images",
    label_dir    = "drive/MyDrive/datasetfinal/labels",
    num_classes  = 2,
    class_names  = ["onstreetlight", "offstreetlight"],
    epochs       = 100,
    imgsz        = 640,
    batch        = 16,
    lr           = 0.01,
    momentum     = 0.9,
    weight_decay = 0.0005,
    save_dir     = "saved_models",
    device       = "cuda" if torch.cuda.is_available() else "cpu",
)
os.makedirs(CFG["save_dir"], exist_ok=True)
RESULTS = []   # global results collector
print("Config OK. Device:", CFG["device"])


Config OK. Device: cuda


## 🌙 Cell 4 — Image Preprocessing Pipeline
**Designed for nighttime streetlight images:**
- Low ambient light → **CLAHE** restores local contrast  
- Overexposed light blobs → careful gamma + clip  
- Motion blur → **Unsharp Mask** recovers edges  
- Tilted capture → rotation **augmentation** during training


In [ ]:
def clahe_enhance(img_bgr, clip=2.0, tile=(8,8)):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)

def gamma_correct(img_bgr, gamma=1.4):
    inv = 1.0/gamma
    table = np.array([(i/255)**inv*255 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(img_bgr, table)

def unsharp_mask(img_bgr, strength=1.3):
    blurred = cv2.GaussianBlur(img_bgr, (0,0), 3)
    return cv2.addWeighted(img_bgr, strength, blurred, -(strength-1), 0)

def preprocess_image(img_bgr):
    img = clahe_enhance(img_bgr)
    img = gamma_correct(img, gamma=1.4)
    img = unsharp_mask(img, strength=1.3)
    return np.clip(img, 0, 255).astype(np.uint8)

def get_train_transforms(imgsz=640):
    return A.Compose([
        A.Rotate(limit=20, p=0.5),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.GaussianBlur(blur_limit=(3,5), p=0.3),
        A.CLAHE(clip_limit=2.0, p=0.4),
        A.RandomGamma(gamma_limit=(80,120), p=0.4),
        A.Resize(imgsz,imgsz),
        A.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]))

print("Preprocessing functions ready")


Preprocessing functions ready


### 🖼️ Cell 4b — Visualise Preprocessing on Sample Image

In [ ]:
def visualise_preprocessing_steps(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Image not found: {img_path}"); return
    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    c1   = cv2.cvtColor(clahe_enhance(img), cv2.COLOR_BGR2RGB)
    c2   = cv2.cvtColor(gamma_correct(clahe_enhance(img)), cv2.COLOR_BGR2RGB)
    c3   = cv2.cvtColor(preprocess_image(img), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 4, figsize=(20,5))
    fig.suptitle("Preprocessing Pipeline for Nighttime Streetlight Images", fontsize=14, fontweight='bold')
    for ax, im, t in zip(axes, [rgb,c1,c2,c3],
                         ["1. Original","2. CLAHE","3. Gamma (1.4)","4. Final (+Unsharp)"]):
        ax.imshow(im); ax.set_title(t, fontweight='bold'); ax.axis('off')
    plt.tight_layout()
    plt.savefig("preprocessing_steps.png", dpi=150, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14,4))
    fig.suptitle("Pixel Intensity Distribution", fontsize=13, fontweight='bold')
    for ax, im, label, color in zip(axes, [rgb,c3], ["Original","Preprocessed"],["#e74c3c","#2ecc71"]):
        ax.hist(im.ravel(), bins=256, color=color, alpha=0.8)
        ax.set_title(label, fontweight='bold')
        ax.axvline(im.ravel().mean(), color='black', linestyle='--',
                   label=f"Mean={im.ravel().mean():.0f}")
        ax.legend()
    plt.tight_layout()
    plt.savefig("intensity_histogram.png", dpi=150, bbox_inches='tight')
    plt.show()

visualise_preprocessing_steps("/mnt/user-data/uploads/1777541693605_frame_0000685.jpg")


Image not found: /mnt/user-data/uploads/1777541693605_frame_0000685.jpg


## 📂 Cell 5 — Dataset Wrapper

In [ ]:
class YOLOFormatDataset(Dataset):
    def __init__(self, img_dir, label_dir, split="train"):
        self.img_paths = sorted(Path(img_dir).glob("*.jpg"))+sorted(Path(img_dir).glob("*.png"))
        self.label_dir = Path(label_dir)

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        ip = self.img_paths[idx]
        lp = self.label_dir / (ip.stem+".txt")
        img = preprocess_image(cv2.imread(str(ip)))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h,w = img.shape[:2]
        boxes, labels = [], []
        if lp.exists():
            for line in lp.read_text().strip().splitlines():
                cls,cx,cy,bw,bh = map(float,line.split())
                boxes.append([(cx-bw/2)*w,(cy-bh/2)*h,(cx+bw/2)*w,(cy+bh/2)*h])
                labels.append(int(cls)+1)
        img = cv2.resize(img,(320,320))
        tensor = transforms.ToTensor()(img)
        return tensor, {
            "boxes" : torch.tensor(boxes,  dtype=torch.float32) if boxes else torch.zeros((0,4)),
            "labels": torch.tensor(labels, dtype=torch.int64)   if labels else torch.zeros(0,dtype=torch.int64),
        }

def collate_fn(batch): return tuple(zip(*batch))
print("Dataset class ready")


Dataset class ready


## 🎯 Cell 6 — YOLO Model Training

In [ ]:
YOLO_MODELS = ["yolo11n.pt"]

def train_yolo_models():
    seen = set()
    for mn in YOLO_MODELS:
        tag = mn.replace(".pt","")
        if tag in seen: continue
        seen.add(tag)
        print(f"\n{'='*50}\nTraining: {mn}\n{'='*50}")
        try:
            model = YOLO(mn)
            model.train(data=CFG["data_yaml"], epochs=CFG["epochs"],
                        imgsz=CFG["imgsz"], batch=CFG["batch"],
                        lr0=CFG["lr"], momentum=CFG["momentum"],
                        weight_decay=CFG["weight_decay"],
                        project="runs/detect", name=f"{tag}_custom", plots=True,
                        hsv_v=0.4, degrees=20, fliplr=0.5, mosaic=1.0, erasing=0.3)
            m  = model.val()
            p  = m.results_dict.get("metrics/precision(B)",0)
            r  = m.results_dict.get("metrics/recall(B)",0)
            m50= m.results_dict.get("metrics/mAP50(B)",0)
            m95= m.results_dict.get("metrics/mAP50-95(B)",0)
            f1 = 2*p*r/(p+r+1e-16)
            best = Path(f"runs/detect/{tag}_custom/weights/best.pt")
            if best.exists():
                import shutil; shutil.copy(best, f"{CFG['save_dir']}/{tag}.pt")
            sz = os.path.getsize(f"{CFG['save_dir']}/{tag}.pt")/1e6 if best.exists() else None
            RESULTS.append(dict(Model=tag, Category="YOLO",
                Precision=round(p,4), Recall=round(r,4),
                mAP50=round(m50,4), mAP50_95=round(m95,4), F1=round(f1,4),
                ModelSizeMB=round(sz,2) if sz else None,
                RPi=tag in ["yolov8n","yolov5nu","yolo11n"]))
            print(f"  mAP50={m50:.4f}  F1={f1:.4f}")
        except Exception as e:
            print(f"[WARN] {mn}: {e}")

train_yolo_models()
print(f"YOLO done. Results collected: {len(RESULTS)}")



Training: yolo11n.pt
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=20, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.3, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=yolo11n_custom, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati